In [4]:
import pandas as pd

In [6]:
## First we need to print the project names for the PyPI packages in our dataset
data = pd.read_csv('../../package_data.csv')

#pypi_projects = data[data['System'] == 'PYPI']['package_name'].unique()

## print the project names surrounded by quotes and separated by commas in one line 
#print(', '.join([f'"{project}"' for project in pypi_projects]))

## The data is collected via Google BQ 
Here is this URL: https://console.cloud.google.com/bigquery?ws=!1m25!1m4!4m3!1scollectingdatadec2025!2spypi_data!3sversion_specific_download_count!1m10!12m5!1m3!1scollectingdatadec2025!2sus-central1!3sd789316d-f467-4f2e-9589-6acbf0ed4c64!2e1!14m3!1scontrol-variables!2sbquxjob_4e6a6134_19b8f73c202!3sUS!1m3!3m2!1scollectingdatadec2025!2spypi_data!1m4!4m3!1scollectingdatadec2025!2spypi_data!3spypi_data_specific_projects&project=control-variables

In [ ]:
## here is the query: 
## NOTE: fill in ' project IN ()' using the PACKAGES printed above 

# CREATE OR REPLACE TABLE `collectingdatadec2025.pypi_data.pypi_data_specific_projects` AS
# SELECT * FROM `bigquery-public-data.pypi.file_downloads` 
# WHERE TIMESTAMP_TRUNC(timestamp, DAY) >= TIMESTAMP("2025-12-29")  AND TIMESTAMP_TRUNC(timestamp, DAY) <= TIMESTAMP("2026-01-04")
# AND project IN ()


# THEN we count the downloads per version
# SELECT
#   project,
#   file.version AS version,
#   COUNT(*) AS download_count
# FROM `collectingdatadec2025.pypi_data.pypi_data_specific_projects`
# GROUP BY project, version
# ORDER BY project, version;

## Verifying number of packages

In [7]:
download_count = pd.read_csv('pypi_download_count_1229_to_0104.csv')

## We have download counts for all packages ✅

We need to merge the PyPI data with the package data.

In [11]:
pypi_data = data[data['System'] == 'PYPI']

In [13]:
import re

def clean_version(tag_name):
    """Remove package name prefixes before version numbers."""
    tag_name = str(tag_name)
    
    # Strategy 1: Find @ followed by a proper version pattern (digit.digit...)
    # Handles: @org/package@1.0.0 -> 1.0.0
    matches = list(re.finditer(r'@(\d+\.\d+[^\s]*)$', tag_name))
    if matches:
        return matches[-1].group(1)
    
    # Strategy 2: Find / followed by version pattern (for package/version style)
    # Handles: package-name/v1.0.0 -> 1.0.0
    match = re.search(r'/[v]?(\d+\.\d+.*)$', tag_name)
    if match:
        return match.group(1)
    
    # Strategy 3: Find _ followed by v and version pattern
    # Handles: gmv3_v2.0.0 -> 2.0.0
    match = re.search(r'_v(\d+\.\d+.*)$', tag_name)
    if match:
        return match.group(1)
    
    # Strategy 4: Handle package-version style with hyphen
    # Handles: package-1.0.0 -> 1.0.0
    match = re.search(r'-(\d+\.\d+.*)$', tag_name)
    if match:
        return match.group(1)
    
    # Strategy 5: Fallback - remove any non-digit prefix (for v1.0.0 style)
    # Handles: v1.0.0 -> 1.0.0
    match = re.search(r'^[^\d]*(\d+.*)$', tag_name)
    if match:
        return match.group(1)
    
    return tag_name

In [14]:
pypi_data['tag_name_clean'] = pypi_data['tag_name'].apply(clean_version)

/var/folders/6f/wmsnc89s2tn1gpcjgm2rbtjr0000gn/T/ipykernel_66216/3699179088.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  pypi_data['tag_name_clean'] = pypi_data['tag_name'].apply(clean_version)


In [17]:
merge = pd.merge(pypi_data, 
                 download_count, 
                 left_on=['package_name', 'tag_name_clean'], 
                 right_on=['package', 'version'], 
                 how='left')

20,101 versions of out 105,831 are missing values. These should be 0.

In [18]:
## add a '0' for versions that had no downloads in that week (NaN values)
merge['download_count'] = merge['download_count'].fillna(0)

In [20]:
## only save the package name, tag_name, and download_count columns
merge[['package_name', 'tag_name', 'download_count']].to_csv('pypi_download_count_1229_to_0104_with_0s.csv', index=False)

In [21]:
test = pd.read_csv('pypi_download_count_1229_to_0104_with_0s.csv')